In [24]:
# Импорты сюдаа
import numpy as np
import matplotlib.pyplot as plt
import segyio
import segyio.tools
from scipy.signal import correlate
from scipy.stats import spearmanr

In [25]:
class RickerWaveletGenerator:
    """
    Генератор синтетических сейсмических трасс на основе свёрточной модели.
    """
    
    def __init__(self, wavelet_type='ricker', wavelet_freq=30, dt=0.002, length=0.128):
        """
        Инициализация генератора.

        Параметры:
        ----------
        wavelet_type : str
                тип вейвлета по умолчанию ('ricker', других пока нет)
        wavelet_freq : float
                доминантная частота вейвлета в Гц (чем выше частота, тем уже импульс)
        dt : float
                интервал дискретизации в секундах (шаг по времени)
        length : float
                полная длина вейвлета в секундах по умолчанию (от -length/2 до length/2)
        """
        self.wavelet_type = wavelet_type
        self.wavelet_freq = wavelet_freq
        self.length = length
        self.dt = dt
        
        self.wavelet_time, self.wavelet = self._create_ricker_wavelet()
    
    def _create_ricker_wavelet(self):
        """
        Создание вейвлета Рикера.

        Формула: (1 - 2π²f²t²) * exp(-π²f²t²)
        где f - доминантная частота, t - время
    
        Возвращает:
        ----------
        wavelet_time : np.ndarray
                временная ось вейвлета (от -length/2 до length/2 с шагом dt)
        wavelet : np.ndarray
                амплитуды вейвлета (не нормированные)
        """
        # Временная сетка для импульса
        t = np.arange(-self.length/2, self.length/2, self.dt) #создаёт массив чисел от -self.length/2 (-0.064) до self.length/2 (0.064) с шагом self.dt (0.002 сек)
        f = self.wavelet_freq
        
        # Формула вейвлета Рикера
        tau = np.pi * f * t
        wavelet = (1 - 2 * tau**2) * np.exp(-tau**2)
        
        # Нормализация (максимальная амплитуда = 1)
        #wavelet = wavelet / np.max(np.abs(wavelet))
        
        return t, wavelet
    
    def get_wavelet(self):
        """
        Возвращает импульс источника (вейвлет). Создаёт его, если ещё не создан, иначе возвращает существующий.
    
        Возвращает:
        ----------
        tuple : (wavelet_time, wavelet)
                временная ось и амплитуды вейвлета
        """
        if self.wavelet is None:
            if self.wavelet_type == 'ricker':
                t, w = self._create_ricker_wavelet()
                self.wavelet_time = t
                self.wavelet = w
            else:
                raise ValueError(f"Неподдерживаемый тип вейвлета: {self.wavelet_type}")
        return self.wavelet_time, self.wavelet
    
    def generate_trace(self, reflectivity_series, noise_level=0.0):
        """
        Генерирует сейсмическую трассу по коэффициентам отражения.
    
        Параметры:
        ----------
        reflectivity_series : np.ndarray
                временной ряд коэффициентов отражения (импульсная характеристика среды); длина массиво определяет длительность трассы в отсчётах
        noise_level : float
                уровень шума (0 = без шума по умолчанию, 0.05 = 5% от RMS сигнала)
    
        Возвращает:
        ----------
        trace : np.ndarray
                синтетическая сейсмическая трасса (результат свёртки)
    
        Алгоритм:
        ----------
        1. Вейвлет
        2. Выполнение свёртки: trace = reflectivity * wavelet
        3. Обрезка до длины исходного ряда
        4. Добавление шума (если необходимо)
        """
        # 1. Импульс источника
        wavelet = self.get_wavelet()
        
        # 2. Свёртка коэффициентов отражения с импульсом
        #    'full' -> полная свёртка, потом обрезаем до длины reflectivity_series
        trace_full = np.convolve(reflectivity_series, wavelet, mode='full')
        
        # 3. Обрезка до нужной длины (сохранение временной оси как у reflectivity_series)
        trace = trace_full[:len(reflectivity_series)]
        
        # 4. Шум (если нужно)
        if noise_level > 0:
            # Оценка RMS сигнала
            rms_signal = np.sqrt(np.mean(trace**2))
            # Генерирация шума
            noise = np.random.randn(len(trace)) * noise_level * rms_signal
            trace = trace + noise
        
        return trace
    
    def plot_wavelet(self):
        """Визуализирует импульс источника (если надо, необязаельно)"""
        wavelet = self.get_wavelet()
        plt.figure(figsize=(5, 2))
        plt.plot(self.wavelet_time, wavelet)
        plt.title(f"Импульс источника по умолчанию ({self.wavelet_type}, freq={self.wavelet_freq} Гц)")
        plt.xlabel("Время (с)")
        plt.ylabel("Амплитуда")
        plt.grid(True)
        plt.show()

In [26]:
class ReflectionCoefficients:
    """
    Расчёт коэффициентов отражения для горизонтально-слоистой среды.
    """
    
    def __init__(self, depths, velocities, densities):
        """    
        Параметры:
        ----------
        depths : np.ndarray
                глубины границ от поверхности (м)
                Например: [500, 1200] — границы на 500м и 1200м - 3 слоя
                Длина = количество границ
        velocities : np.ndarray
                скорости в слоях (м/с)
                Длина = len(depths) + 1 (слой над первой границей + слои между границами + последний слой)
        densities : np.ndarray
                плотности в слоях (кг/м³)
                Длина = len(depths) + 1
        
        Действия:
        ----------
        1. Сохранение входных параметров
        2. Вычисление акустических импедансов слоёв
        3. Вычисление коэффициентов отражения на границах
        4. Вычисление двустороннего времени прихода отражений
        """
        self.depths = np.array(depths)
        self.velocities = np.array(velocities)
        self.densities = np.array(densities)
        self.num_layers = len(velocities)
        self.num_boundaries = len(depths)
        
        # Расчёт
        self.impedances = self._compute_impedances()
        self.reflection_coeffs = self._compute_reflection_coeffs()
        self.two_way_times = self._compute_times()
    
    def _compute_impedances(self):
        """
        Вычисляет акустический импеданс для каждого слоя.
    
        Формула: Z = ρ * V
        где ρ - плотность, V - скорость
    
        Возвращает:
        ----------
        impedances : np.ndarray
                массив импедансов для всех слоёв (длина = количество слоёв)
        """
        return self.densities * self.velocities
    
    def _compute_reflection_coeffs(self):
        """
        Вычисляет коэффициент отражения на каждой границе.
    
        Формула: R = (Z₂ - Z₁) / (Z₂ + Z₁)
        где Z₁ - импеданс верхнего слоя, Z₂ - импеданс нижнего слоя
    
        Возвращает:
        ----------
        R : np.ndarray
                массив коэффициентов отражения (длина = количество границ)
            
        Примечание:
        ----------
        R > 0 - отражение положительной полярности (Z₂ > Z₁)
        R < 0 - отражение отрицательной полярности (Z₂ < Z₁)
        """
        R = []
        for i in range(self.num_boundaries):
            Z_upper = self.impedances[i]
            Z_lower = self.impedances[i+1]
            coeff = (Z_lower - Z_upper) / (Z_lower + Z_upper)
            R.append(coeff)
        return np.array(R)
    
    def _compute_times(self):
        """
        Вычисляет двустороннее время прихода отражённой волны от каждой границы.
    
        Принцип:
        ----------
        Время = 2 * (толщина_слоя / скорость_слоя)
        Для каждой границы суммируется время прохождения всех вышележащих слоёв
    
        Возвращает:
        ----------
        times : np.ndarray
            массив времён для каждой границы (в секундах)
        """
        times = []
        cumulative = 0.0
        
        for i in range(self.num_boundaries):
            if i == 0:
                thickness = self.depths[0]
            else:
                thickness = self.depths[i] - self.depths[i-1]
            
            cumulative += 2 * thickness / self.velocities[i]
            times.append(cumulative)
        
        return np.array(times)
    
    def get_reflectivity_series(self, dt, total_time):
        """
        Преобразует коэффициенты отражения в дискретный временной ряд.
    
        Параметры:
        ----------
        dt : float
                интервал дискретизации в секундах
        total_time : float
                полное время записи в секундах
    
        Возвращает:
        ----------
        time_axis : np.ndarray
                временная ось (от 0 до total_time с шагом dt)
        reflectivity_series : np.ndarray
                временной ряд коэффициентов отражения (импульсы на временах границ)
    
        Примечание:
        ----------
        Коэффициенты отражения размещаются на ближайших временных отсчётах с помощью округления (int(round(t / dt)))
        """
        num_samples = int(total_time / dt) + 1
        time_axis = np.arange(0, total_time + dt, dt)[:num_samples]
        reflectivity = np.zeros(num_samples)
        
        for t, r in zip(self.two_way_times, self.reflection_coeffs):
            idx = int(round(t / dt))
            if idx < num_samples:
                reflectivity[idx] = r
        
        return time_axis, reflectivity
    
    def print_summary(self):
        """Вывод геологической модели в консоль (необязательно)"""
        print("ГЕОЛОГИЧЕСКАЯ МОДЕЛЬ\n")
        
        # Заголовок таблицы
        print(f"{'Слой':<6} {'V (м/с)':<10} {'ρ (кг/м³)':<12} {'Z (кг/(м²·с))':<18} {'Толщина':<12}")
        print("-" * 70)
        
        # Слой 0 (до первой границы)
        print(f"{'0':<6} {self.velocities[0]:<10.0f} {self.densities[0]:<12.1f} {self.impedances[0]:<18.2f} {self.depths[0]:<12.1f} м")
        
        # Промежуточные слои
        for i in range(1, self.num_layers - 1):
            thickness = self.depths[i] - self.depths[i-1]
            print(f"{i:<6} {self.velocities[i]:<10.0f} {self.densities[i]:<12.1f} {self.impedances[i]:<18.2f} {thickness:<12.1f} м")
        
        # Последний слой (полубесконечный)
        if self.num_layers > 1:
            last_thickness = "∞"
            print(f"{self.num_layers-1:<6} {self.velocities[-1]:<10.0f} {self.densities[-1]:<12.1f} {self.impedances[-1]:<18.2f} {last_thickness:<12}")
        
        print("ГРАНИЦЫ\n")
        for i in range(self.num_boundaries):
            sign = "+" if self.reflection_coeffs[i] >= 0 else "-"
            print(f"Граница {i+1}: глубина {self.depths[i]} м, R = {self.reflection_coeffs[i]:.4f} ({sign}), t = {self.two_way_times[i]:.4f} с")


In [27]:
class SeismicTraceGenerator:
    """
    Генератор синтетической трассы.
    """
    
    def __init__(self, wavelet_freq=30, dt=0.002, noise_level=0.0):
        """    
        Создаёт вейвлет с заданными параметрами для последующей свёртки.
        
        Параметры:
        ----------
        wavelet_freq : float
                доминантная частота вейвлета в Гц
        dt : float
                интервал дискретизации в секундах
        noise_level : float
                уровень шума (0 = без шума, 0.05 = 5% от RMS сигнала)
        """
        self.wavelet_freq = wavelet_freq
        self.dt = dt
        self.noise_level = noise_level
        
        # Создание вейвлета с помощью класса RickerWaveletGenerator()
        wavelet = RickerWaveletGenerator(wavelet_freq=wavelet_freq, dt=dt, length=0.2)
        self.wavelet_time, self.wavelet = wavelet.get_wavelet()
    
    def _add_noise(self, trace):
        """
        Добавляет шум к трассе с уровнем относительно RMS сигнала.
    
        Параметры:
        ----------
        trace : np.ndarray
                входная трасса без шума
    
        Возвращает:
        ----------
        trace_with_noise : np.ndarray
                трасса с добавленным гауссовским шумом
        """
        if self.noise_level <= 0:
            return trace
        
        rms = np.sqrt(np.mean(trace**2))
        if rms > 0:
            noise = np.random.randn(len(trace)) * self.noise_level * rms
            return trace + noise
        return trace
    
    def generate_from_reflectivity(self, reflectivity_series):
        """
        Генерация трассы из временного ряда коэффициентов отражения.
    
        Параметры:
        ----------
        reflectivity_series : np.ndarray
                временной ряд коэффициентов отражения из 
    
        Возвращает:
        ----------
        trace : np.ndarray
                сейсмическая трасса (результат свёртки с вейвлетом)
    
        Примечание:
        ----------
        Используется mode='full' для точного позиционирования импульса,
        затем результат обрезается до длины исходного ряда.
        """
        # Полная свёртка
        trace_full = np.convolve(reflectivity_series, self.wavelet, mode='full')
        
        # Обрезаем до длины исходного ряда
        trace = trace_full[:len(reflectivity_series)]
        
        # Добавляем шум
        trace = self._add_noise(trace)
        
        return trace
    
    def generate_from_model(self, depths, velocities, densities, total_time):
        """
        Полный конвейер: геология → трасса. ГЛАВНАЯ ФУНКЦИЯ ГЕНЕРАЦИИ ТРАССЫ
        
        Возвращает:
        time_axis, trace, rc_model
        """
        # Шаг 1: коэффициенты отражения
        rc = ReflectionCoefficients(depths, velocities, densities)
        
        # Шаг 2: временной ряд коэффициентов
        time_axis, reflectivity = rc.get_reflectivity_series(self.dt, total_time)
        
        # Шаг 3: свёртка с вейвлетом
        trace = self.generate_from_reflectivity(reflectivity)
        
        return time_axis, trace, rc
    
    def visualize_full(self, depths, velocities, densities, total_time):
        """
        Комплексная визуализация всей цепочки (необязательно):
        1. Скоростная модель
        2. Импедансы слоёв
        3. Коэффициенты отражения
        4. Синтетическая трасса
        """
        # Генерируем трассу
        time_axis, trace, rc = self.generate_from_model(depths, velocities, densities, total_time)
        
        # Получаем временной ряд коэффициентов
        _, reflectivity = rc.get_reflectivity_series(self.dt, total_time)
        
        # Создаём фигуру из 4 подграфиков
        fig, axes = plt.subplots(4, 1, figsize=(10, 8))
        
        # 1. Скоростная модель
        depths_plot = [0] + list(depths)
        velocities_plot = velocities
        axes[0].step(depths_plot + [depths_plot[-1] + 500], 
                     velocities_plot + [velocities_plot[-1]], 
                     where='post', linewidth=2)
        axes[0].set_ylabel("Скорость V, м/с")
        axes[0].set_xlabel("Глубина, м")
        #axes[0].invert_xaxis()
        axes[0].grid(True)
        axes[0].set_title("1. Скоростная модель")
        
        # 2. Импедансы слоёв
        impedances = rc.impedances
        layer_names = [f"Слой {i+1}" for i in range(len(impedances))]
        axes[1].bar(layer_names, impedances, color='lightblue', edgecolor='black')
        axes[1].set_ylabel("Акустический импеданс Z = ρ·V")
        axes[1].set_title("2. Импедансы слоёв")
        axes[1].grid(True, axis='y')
        
        # 3. Коэффициенты отражения
        R = rc.reflection_coeffs
        boundary_names = [f"Гр.{i+1}" for i in range(len(R))]
        colors = ['green' if r >= 0 else 'red' for r in R]
        axes[2].bar(boundary_names, R, color=colors, edgecolor='black')
        axes[2].axhline(y=0, color='black', linewidth=1)
        axes[2].set_ylabel("Коэффициент отражения R")
        axes[2].set_title("3. Коэффициенты отражения на границах")
        axes[2].grid(True, axis='y')
        
        # 4. Синтетическая трасса
        axes[3].plot(time_axis, trace, 'b-', linewidth=1.5)
        axes[3].set_xlabel("Время, с")
        axes[3].set_ylabel("Амплитуда")
        axes[3].set_title("4. Синтетическая сейсмическая трасса")
        axes[3].grid(True)
        
        # Отмечаем времена прихода границ
        for t in rc.two_way_times:
            axes[3].axvline(x=t, color='red', linestyle='--', alpha=0.5, linewidth=1)
        
        plt.tight_layout()
        plt.show()
        
        return time_axis, trace, rc


In [28]:
# ПОШЛА ПОПЫТКА

In [29]:
def generate_synthetic_section(num_traces, depths_base, velocities, densities, 
                               horizon_index=1, variation_amplitude=50, 
                               variation_type='sinusoidal',
                               wavelet_freq=30, dt=0.001, total_time=2.0, 
                               noise_level=0.01):
    """
    Генерация синтетического сейсмического разреза с латеральными изменениями (генерация множества трасс по введенным вручную данным)
    
    Параметры:
    ----------
    num_traces : int
        количество трасс в разрезе
    depths_base : list
        базовые глубины границ (м) [граница1, граница2, ...]
    velocities : list
        скорости в слоях (м/с) [слой0, слой1, ...] (len = len(depths_base) + 1)
    densities : list
        плотности в слоях (кг/м³) [слой0, слой1, ...] (len = len(depths_base) + 1)
    horizon_index : int
        индекс горизонта (границы), который будет меняться (0 - первая граница)
    variation_amplitude : float
        амплитуда изменения глубины (м)
    variation_type : str
        тип изменения: 'sinusoidal', 'linear', 'random'
    wavelet_freq : float
        доминантная частота вейвлета (Гц)
    dt : float
        интервал дискретизации (с)
    total_time : float
        полное время записи (с)
    noise_level : float
        уровень шума (0 = без шума)
    
    Возвращает:
    ----------
    traces : np.ndarray
        массив трасс размером (num_traces, num_samples)
    time_axis : np.ndarray
        временная ось
    depths_array : np.ndarray
        массив глубин каждой трассы для изменяющегося горизонта
    """
    
    num_samples = int(total_time / dt) + 1
    time_axis = np.arange(0, total_time + dt, dt)[:num_samples]
    
    # Создание генератора трасс
    trace_gen = SeismicTraceGenerator(wavelet_freq=wavelet_freq, dt=dt, noise_level=noise_level)
    
    # Массив глубин для каждой трассы (рисунок вейвлетов отклонится от прямой)
    x_positions = np.arange(num_traces)
    
    if variation_type == 'sinusoidal':
        # Синусоидальное изменение глубины вдоль профиля
        variation = variation_amplitude * np.sin(2 * np.pi * x_positions / num_traces)
    elif variation_type == 'linear':
        # Линейное изменение
        variation = variation_amplitude * (2 * x_positions / num_traces - 1)
    elif variation_type == 'random':
        # Случайное изменение с плавностью (скользящее среднее)
        variation = np.random.randn(num_traces) * variation_amplitude / 2
        # Сглаживание
        from scipy.ndimage import uniform_filter1d
        variation = uniform_filter1d(variation, size=5)
    else:
        variation = np.zeros(num_traces)
    
    # Глубины для каждой трассы
    depths_array = []
    traces = []
    
    for i in range(num_traces):
        # Копия базовых глубин
        depths_i = depths_base.copy()
        # Изменение выбранного горизонта
        depths_i[horizon_index] = depths_base[horizon_index] + variation[i]
        depths_array.append(depths_i[horizon_index])
        
        # Генерация трасс
        time_axis_i, trace_i, rc_i = trace_gen.generate_from_model(
            depths_i, velocities, densities, total_time
        )
        traces.append(trace_i)
    
    traces = np.array(traces)
    depths_array = np.array(depths_array)
    
    return traces, time_axis, depths_array

In [30]:
depths_base = [300, 700, 1000, 1800]  # 4 границы - 5 слоёв
velocities = [1000, 2200, 1600, 2000, 3500]  # м/с
densities = [1000, 2200, 1400, 2600, 2700]    # кг/м³

num_traces = 100
dt=0.001
horizon_index=2
variation_amplitude=400
variation_type='random'
wavelet_freq=30
total_time=3.0
noise_level=0.1

print("\nГеологическая модель:")
print(f"  Границы на глубинах: {depths_base} м")
print(f"  Скорости в слоях: {velocities} м/с")
print(f"  Плотности в слоях: {densities} кг/м³")


Геологическая модель:
  Границы на глубинах: [300, 700, 1000, 1800] м
  Скорости в слоях: [1000, 2200, 1600, 2000, 3500] м/с
  Плотности в слоях: [1000, 2200, 1400, 2600, 2700] кг/м³


In [31]:
traces, time_axis, depths_changed = generate_synthetic_section(
    num_traces=num_traces,
    depths_base=depths_base,
    velocities=velocities,
    densities=densities,
    horizon_index=horizon_index,  # Меняем третью границу (1000 м) - horizon_index = 2
    variation_amplitude=variation_amplitude,  # Амплитуда ±80 м
    variation_type=variation_type,
    wavelet_freq=wavelet_freq,
    dt=dt,
    total_time=total_time,
    noise_level=noise_level
)

In [32]:
print(f"\nСгенерировано {num_traces} трасс")
print(f"  Отсчётов на трассу: {traces.shape[1]}")
print(f"  Интервал: {dt*1000} мс")
print(f"  Изменение глубины горизонта: от {depths_changed.min():.1f} до {depths_changed.max():.1f} м")


Сгенерировано 100 трасс
  Отсчётов на трассу: 3001
  Интервал: 1.0 мс
  Изменение глубины горизонта: от 735.1 до 1209.8 м


In [33]:
#print([attr for attr in dir(segyio.TraceField) if not attr.startswith('_')])


In [34]:
def save_traces_to_sgy(traces, dt_ms, filename="synthetic_section.sgy", 
                       trace_numbers=None, title="Synthetic Section"):
    """
    Сохранение массива трасс в SEG-Y файл.
    
    Параметры:
    ----------
    traces : np.ndarray
        массив трасс размером (num_traces, num_samples)
    dt_ms : float
        интервал дискретизации в миллисекундах
    filename : str
        имя выходного файла
    trace_numbers : list или np.ndarray, optional
        номера трасс (если None, будут 1, 2, 3, ...)
    title : str
        текстовое описание для заголовка
    
    Возвращает:
    ----------
    filename : str
        имя созданного файла
    """
    
    num_traces, num_samples = traces.shape
    dt_us = int(dt_ms * 1000)
    
    # Создание спецификации файла
    spec = segyio.spec()
    spec.ilines = range(1, num_traces + 1)
    spec.xlines = [1]
    spec.samples = range(num_samples)
    spec.format = 5
    
    with segyio.create(filename, spec) as f:
        
        # Бинарный заголовок
        f.bin[segyio.BinField.Interval] = dt_us
        f.bin[segyio.BinField.Samples] = num_samples
        f.bin[segyio.BinField.Format] = 5
        
        # Текстовый заголовок
        text_header = []
        text_header.append(f"C 1 {title}")
        text_header.append(f"C 2 Created by synthetic seismic generator")
        text_header.append(f"C 3 Number of traces: {num_traces}")
        text_header.append(f"C 4 Samples per trace: {num_samples}")
        text_header.append(f"C 5 Sample interval: {dt_ms} ms ({dt_us} us)")
        text_header.append(f"C 6 Data format: IEEE 32-bit float")
        text_header.append(f"C 7 Date: {np.datetime64('today')}")
        
        while len(text_header) < 40:
            text_header.append(" " * 80)
        
        text_bytes = "\n".join(text_header)[:3200].encode('ascii', errors='ignore')
        f.text[0] = text_bytes.ljust(3200, b' ')
        
        # Трассы и заголовки
        for i in range(num_traces):
            trace_num = i + 1 if trace_numbers is None else trace_numbers[i]
            
            # Данные трассы
            f.trace[i] = traces[i].astype(np.float32)
            
            # Заголовок трассы (существующие поля)
            f.header[i][segyio.TraceField.TraceNumber] = trace_num
            f.header[i][segyio.TraceField.TraceIdentifier] = 1  # 1 = живая трасса
            f.header[i][segyio.TraceField.CDP] = trace_num      # номер CDP (можно использовать как номер трассы)
            f.header[i][segyio.TraceField.CDP_TRACE] = i + 1    # трасса в CDP
            f.header[i][segyio.TraceField.TRACE_SEQUENCE_LINE] = i + 1
            f.header[i][segyio.TraceField.TRACE_SEQUENCE_FILE] = i + 1
    
    print(f"\nФайл сохранён: {filename}")
    print(f"  Трасс: {num_traces}")
    print(f"  Отсчётов на трассу: {num_samples}")
    print(f"  Интервал: {dt_ms} мс ({dt_us} мкс)")
    
    return filename

In [14]:
def save_section_with_metadata(traces, dt, filename="synthetic_section.sgy", 
                                depths_array=None, title="Synthetic Section"):
    """
    Сохранение разреза в SEG-Y с дополнительной информацией. ЭТУ НЕ ПРОВЕРЯЛА
    
    Параметры:
    ----------
    traces : np.ndarray
        Массив трасс
    dt : float
        Интервал дискретизации в СЕКУНДАХ
    filename : str
        Имя файла
    depths_array : np.ndarray, optional
        Массив глубин для записи в заголовок трасс (как вспомогательная информация)
    title : str
        Заголовок
    """
    
    # Конвертируем секунды в миллисекунды
    dt_ms = dt * 1000
    
    num_traces, num_samples = traces.shape
    
    # Создаём спецификацию
    spec = segyio.spec()
    spec.ilines = range(1, num_traces + 1)
    spec.xlines = [1]
    spec.samples = range(num_samples)
    spec.format = 5
    
    with segyio.create(filename, spec) as f:
        
        # Бинарный заголовок
        f.bin[segyio.BinField.Interval] = int(dt_ms * 1000)  # в микросекундах
        f.bin[segyio.BinField.Samples] = num_samples
        f.bin[segyio.BinField.Format] = 5
        
        # Текстовый заголовок
        text_lines = [
            f"C 1 {title}",
            f"C 2 Traces: {num_traces}, Samples: {num_samples}",
            f"C 3 DT: {dt_ms} ms",
            f"C 4 Generated by Python/segyio",
        ]
        if depths_array is not None:
            text_lines.append(f"C 5 Horizon depth varies from {depths_array.min():.1f} to {depths_array.max():.1f} m")
        
        while len(text_lines) < 40:
            text_lines.append(" " * 80)
        
        text_bytes = "\n".join(text_lines)[:3200].encode('ascii', errors='ignore')
        f.text[0] = text_bytes.ljust(3200, b' ')
        
        # Трассы
        for i in range(num_traces):
            f.trace[i] = traces[i].astype(np.float32)
            
            # Заполняем заголовок трассы
            f.header[i][segyio.TraceField.TraceNumber] = i + 1
            f.header[i][segyio.TraceField.TraceIdentifier] = 1
            
            # Если переданы глубины, записываем их в поле для вспомогательной информации
            # (используем поле Coordinate Scale или EnergyLossProduct для хранения глубины)
            if depths_array is not None:
                # Сохраняем глубину как масштабированное целое
                depth_scaled = int(depths_array[i] * 10)  # масштабируем для целого числа
                f.header[i][segyio.TraceField.CDP_X] = depth_scaled  # используем CDP_X для хранения глубины
                f.header[i][segyio.TraceField.CDP_Y] = 0
    
    print(f"\nСохранён файл: {filename}")
    print(f"  Размер: {num_traces} x {num_samples}")
    print(f"  dt = {dt_ms} мс")
    
    return filename

In [35]:
dt_ms = dt * 1000  # dt = 0.001 сек → 1 мс

#Сохранение сгенерированного
save_traces_to_sgy(traces, dt_ms, filename="multilayer_section.sgy",
                   title="5-layer model with sinusoidal variation")


Файл сохранён: multilayer_section.sgy
  Трасс: 100
  Отсчётов на трассу: 3001
  Интервал: 1.0 мс (1000 мкс)


'multilayer_section.sgy'

In [ ]:
#=======================================================================================================================================

In [ ]:
#=======================================================================================================================================

In [ ]:
#=======================================================================================================================================

In [136]:
# ПРОГА ОРИЕНТИРУЕТСЯ ОТ ВРЕМЕНИ, НАДО ОТ ОТСЧЕТОВ (на потом)

In [107]:
'''
def generate_section_from_model(num_traces, depths_base, velocities, densities,
                                 horizon_to_vary=1, amplitude=50, 
                                 wavelet_freq=30, dt=0.001, total_time=2.0, 
                                 noise_level=0.0):
    """
    Генерация разреза с использованием классов из try_1.2.
    """
    num_samples = int(total_time / dt) + 1
    time_axis = np.arange(0, total_time + dt, dt)[:num_samples]
    
    # Инициализация генератора
    generator = SeismicTraceGenerator(wavelet_freq=wavelet_freq, dt=dt, noise_level=noise_level)
    
    traces = []
    depths_variation = []
    
    for i in range(num_traces):
        # Создаём вариацию глубины
        variation = amplitude * np.sin(2 * np.pi * i / num_traces)
        
        depths_i = depths_base.copy()
        depths_i[horizon_to_vary] = depths_base[horizon_to_vary] + variation
        
        depths_variation.append(depths_i[horizon_to_vary])
        
        # Генерируем трассу
        _, trace, _ = generator.generate_from_model(depths_i, velocities, densities, total_time)
        traces.append(trace)
    
    return np.array(traces), time_axis, np.array(depths_variation)


# Быстрый тест
traces_section, time_axis_section, depths_var = generate_section_from_model(
    num_traces=80,
    depths_base=[200, 500, 900, 1400, 2000],
    velocities=[1500, 2000, 2500, 3000, 3500, 4000],
    densities=[1800, 2100, 2300, 2500, 2600, 2700],
    horizon_to_vary=2,  # меняем третью границу
    amplitude=60,
    wavelet_freq=35,
    dt=0.001,
    total_time=2.5,
    noise_level=0.01
)

# Визуализация результата
plt.figure(figsize=(12, 8))
plt.imshow(traces_section.T, aspect='auto', cmap='seismic', 
           extent=[0, 80, time_axis_section[-1], 0])
plt.colorbar(label='Амплитуда')
plt.xlabel('Номер трассы')
plt.ylabel('Время (с)')
plt.title('Сейсмический разрез с изменяющейся глубиной горизонта')
plt.show()
'''

'\ndef generate_section_from_model(num_traces, depths_base, velocities, densities,\n                                 horizon_to_vary=1, amplitude=50, \n                                 wavelet_freq=30, dt=0.001, total_time=2.0, \n                                 noise_level=0.0):\n    """\n    Генерация разреза с использованием классов из try_1.2.\n    """\n    num_samples = int(total_time / dt) + 1\n    time_axis = np.arange(0, total_time + dt, dt)[:num_samples]\n\n    # Инициализация генератора\n    generator = SeismicTraceGenerator(wavelet_freq=wavelet_freq, dt=dt, noise_level=noise_level)\n\n    traces = []\n    depths_variation = []\n\n    for i in range(num_traces):\n        # Создаём вариацию глубины\n        variation = amplitude * np.sin(2 * np.pi * i / num_traces)\n\n        depths_i = depths_base.copy()\n        depths_i[horizon_to_vary] = depths_base[horizon_to_vary] + variation\n\n        depths_variation.append(depths_i[horizon_to_vary])\n\n        # Генерируем трассу\n  